# CymbalGoal AlloyDB Workshop—Data Spike: Explore and Size

**Track A (AlloyDB), pipeline step 00.** A read-only look at the CymbalGoal data we already have,
done before any generation work, so the corpus approach and the lab slice size come from measured
numbers instead of the outline's guesses.

| | |
| :-- | :-- |
| **Reads** | `gs://class-demo/alloydb-labs/cymbalgoal/checkpoints/` and, as a fallback for the relational tables, the staged `*.csv.gz` one level up |
| **Writes** | the local runtime only (`/content/cymbalgoal-sizing/`)—**nothing** goes to the bucket |
| **Calls** | no Gemini, no Vertex, no AlloyDB. Every cost and load-time figure is arithmetic |
| **Conventions** | config, load, and scope follow `cymbalgoal_de_pipeline_part1.ipynb` / `part2.ipynb` |

### ⚠️ The bucket path is read-only

Another team's labs (mkt013/014/015) load from `gs://class-demo/alloydb-labs/cymbalgoal/`. Every
`gcloud storage` call in this notebook goes through one guard function, `gcs()`, which allows only
`ls` and `cat` and refuses anything else. The last section prints an audit of every call made.

### How to run

**Run all.** Allow 3–6 minutes, most of it reading `appearances` and `game_events`. Every section
ends with a **📋 SEND BACK** block, and the last section, **12 · Sizing handback**, prints all of
those blocks together in one place, ready to copy.

### Sections

| | Section | Question it answers |
| :-- | :-- | :-- |
| 1 | Configuration | the pin, the scope, every knob and price |
| 2 | Bucket inventory | what is actually under `checkpoints/` |
| 3 | Load the tables | eight-table load, row counts against Part 2's `EXPECTED_ROWS`, scope check |
| 4 | Player-seasons in scope | how many per-season reports the data supports (outline promised 150K–250K) |
| 5 | Scouting perspectives | reports if 2 or 3 clubs scout each player-season |
| 6 | Extended absences | mid-season gaps while the club kept playing |
| 7 | Transfer graph | moves per season, busiest clubs, multi-hop chains |
| 8 | Players who share a name | the Paulinho problem, in player-seasons |
| 9 | Size math at 3072-dim float32 | bytes per row, corpus and slice totals, load-time estimate |
| 10 | Report-generation cost | `gemini-3.8-flash` batch pricing, arithmetic only |
| 11 | History tables | rows and Parquet size for appearances, game events, games |
| 12 | Sizing handback | everything above in one paste-able block, plus the read-only audit |

---
## 1 · Configuration

Same pin and scope as Part 1 and Part 2. Everything tunable is in the second cell below; nothing
further down hardcodes a path, a threshold, or a price.

In [ ]:
# Colab Enterprise images vary; make the two hard dependencies explicit (Part 1 convention).
import importlib, subprocess, sys
for mod, pkg in [("pyarrow", "pyarrow"), ("pandas", "pandas")]:
    if importlib.util.find_spec(mod) is None:
        print(f"installing {pkg}…")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("dependencies ok")

In [ ]:
import csv, gzip, io, json, math, os, re, subprocess, sys, time, unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 60)

# ---- Provenance (the Part 1 / Part 2 pin) -------------------------------------------------
SNAPSHOT_DATE   = "2026-08-14"
SNAPSHOT_SHA256 = "3e6742a9c5002b202eeb8803a123bbc2d9c13ae582eb45ffb6f587843693fffa"

# ---- Where the data lives. READ-ONLY. -----------------------------------------------------
GCS_PREFIX = "gs://class-demo/alloydb-labs/cymbalgoal"
CKPT_GCS   = f"{GCS_PREFIX}/checkpoints"
READ_ONLY_PREFIX = GCS_PREFIX          # gcs() refuses any URI outside this, and any write verb

# ---- Local scratch (runtime only; never mirrored to the bucket) ---------------------------
BASE = Path("/content/cymbalgoal-sizing")
try:
    BASE.mkdir(parents=True, exist_ok=True)
except OSError:
    BASE = Path.home() / "cymbalgoal-sizing"
    BASE.mkdir(parents=True, exist_ok=True)
MEASURE_DIR = BASE / "parquet-measure"   # section 11 writes Parquet here to measure size
MEASURE_DIR.mkdir(exist_ok=True)

# ---- Scope (S-21 / S-23, from Part 1) -----------------------------------------------------
BIG5 = ["GB1", "ES1", "IT1", "L1", "FR1"]
# UEFA ids are discovered from the data (section 3), never hardcoded—Part 1 convention.

# Row counts the staged tables must have (Part 2, section 3). A mismatch is flagged, not fatal:
# this notebook is exploring, and a different checkpoint should be reported, not hidden.
EXPECTED_ROWS = {
    "competitions": 65,
    "clubs": 796,
    "players": 13_439,
    "games": 29_740,
    "appearances": 832_193,
    "game_events": 417_617,
    "player_valuations": 297_822,
    "transfers": 65_494,
}
# player_valuations is not needed for any measurement here, so it is not loaded.
TABLES_NEEDED = ["competitions", "clubs", "players", "games",
                 "appearances", "game_events", "transfers"]

# ---- 4 · Player-seasons --------------------------------------------------------------------
MINUTE_THRESHOLDS = [1, 90, 450, 900]          # total scope minutes in the season
OUTLINE_PROMISE   = (150_000, 250_000)         # what the outline said
SUSPECTED_RANGE   = (60_000, 75_000)           # what Patrick suspects
PARTIAL_SEASON_SHARE = 0.60                    # a season with < 60% of the median games is partial

# ---- 5 · Scouting perspectives -------------------------------------------------------------
SCOUTS_PER_PS = [1, 2, 3]
LATER_SIGNER_HORIZON_YEARS = 2    # secondary count: signed within 2 summers of the season's end

# ---- 6 · Extended absences -----------------------------------------------------------------
ABSENCE_MIN_APPS   = 5            # the player needs this many apps for that club that season
ABSENCE_GAP_GAMES  = [4, 6, 10]   # consecutive club matches missed (scope competitions)
ABSENCE_GAP_DAYS   = [30, 60, 90]
ABSENCE_PRIMARY    = (6, 30)      # the headline definition: >= 6 matches AND >= 30 days

# ---- 7 · Transfer graph --------------------------------------------------------------------
NON_CLUB_RE = r"^\s*(?:without club|unknown|retired|career break|ban|vereinslos|no club|\?|-)\s*$"
RESERVE_RE  = r"(?:\bU-?\d{2}\b|\bYouth\b|\bJuvenil|\bPrimavera\b|\bCastilla\b|\bReserves?\b|\s(?:B|II)$)"
EXCLUDE_RESERVE_SIDES_FROM_CHAINS = True
CHAIN_EXAMPLES = 10

# ---- 9 · Size math -------------------------------------------------------------------------
EMBED_DIMS      = 3072            # LOCKED by Lab 1's in-SQL embedding()—see Part 2 README
BYTES_PER_FLOAT = 4               # float32 (pgvector `vector`)
REPORT_WORDS    = (120, 180)
DEFAULT_BYTES_PER_WORD = 6.4      # replaced by a measurement from the existing profiles
REPORT_META_BYTES = 40            # report_id, player_id, season, scout_club_id, perspective, date
# PostgreSQL storage constants (8 KB pages)
PAGE_BYTES, PAGE_HDR, TUPLE_HDR, LINE_PTR = 8192, 24, 24, 4
TOAST_CHUNK      = 1996           # TOAST_MAX_CHUNK_SIZE for 8 KB pages
TOAST_TUPLE_OVH  = TUPLE_HDR + LINE_PTR + 4 + 4 + 4   # chunk_id, chunk_seq, varlena header
TOAST_IDX_PER_CHUNK = 20          # btree entry on (chunk_id, chunk_seq)
TOAST_POINTER    = 18
SCANN_BYTES_PER_ROW_OVH = 64      # ScaNN sq8 = 1 byte/dim + per-row overhead (estimate)
BTREE_BYTES_PER_ROW = 20          # one btree index entry incl. line pointer
REPORT_BTREES = 2                 # PK + one FK index on player_id

# Load-time anchors, measured for Lab 1 (cymbalgoal-intelligent-search/terraform/main.tf):
#   * cluster create ~9 min of a ~20 min Start Lab cold start
#   * student-notebook load ~4.5 min: 8 relational tables + 14,235 profile rows with 3072-dim
#     vectors, COPY through the AlloyDB Python Connector from Colab
#   * ScaNN build on players (13,439 rows, sq8, 8 vCPU): 27 s on the test rig, 47 s in the lab
COLD_START_MIN          = 20
CLUSTER_MIN             = 9
ANCHOR_NOTEBOOK_LOAD_S  = 270
ANCHOR_VECTOR_ROWS      = 14_235
ANCHOR_RELATIONAL_S     = (60, 120)   # ASSUMED split of the 4.5 min—it was measured whole
ANCHOR_SCANN_S          = (27, 47)
ANCHOR_SCANN_ROWS       = 13_439
LOAD_ROWS_PER_S_OVERRIDE = None       # set this once a real COPY rate has been measured

# ---- 10 · Generation cost (arithmetic only) -----------------------------------------------
GEN_MODEL              = "gemini-3.8-flash"
PRICE_IN_PER_M         = 0.75     # promotional list price, USD per 1M tokens
PRICE_OUT_PER_M        = 3.75     # reasoning tokens bill at the output rate
BATCH_DISCOUNT         = 0.50     # batch = half price
PRICE_POSTPROMO        = (1.50, 7.50)   # Part 2 notes the 3.7-flash promo doubles 2027-01-01
TOKENS_PER_WORD_DEFAULT        = 1.35
INPUT_TOKENS_PER_REPORT_DEFAULT = 700   # fact block + instructions; replaced if measurable
REASONING_TOKENS_DEFAULT       = 0
PRICE_EMBED_PER_M      = 0.15     # gemini-embedding-001, from Part 2's config (list price)
PRICE_GROUNDING_PER_1K = 14.00    # Part 2's rate—only shown as a "what if"

print(f"snapshot  {SNAPSHOT_DATE}  {SNAPSHOT_SHA256[:16]}…")
print(f"reading   {CKPT_GCS}/  (read-only)")
print(f"scratch   {BASE}")
print(f"scope     Big 5 = {BIG5}  + UEFA (discovered in section 3)")
print(f"vectors   {EMBED_DIMS} dims x float32 = {EMBED_DIMS * BYTES_PER_FLOAT:,} bytes raw")
print(f"pricing   {GEN_MODEL}  ${PRICE_IN_PER_M}/${PRICE_OUT_PER_M} per 1M, batch x{BATCH_DISCOUNT}")

### 1.1 The read-only guard

Every bucket call in this notebook goes through `gcs()`. It accepts only `ls` and `cat`, only for
URIs under `gs://class-demo/alloydb-labs/cymbalgoal/`, and logs each call for the audit in section 12.
There is no code path in this notebook that can copy, move, or delete an object.

`CG_SIZING_FIXTURE` is an offline test hook: if set, the same guard serves a local directory laid
out like the bucket. It is unset in Colab.

In [ ]:
GCS_LOG = []                      # every call, for the section 12 audit
_READ_VERBS = {"ls", "cat"}
_FIXTURE = os.environ.get("CG_SIZING_FIXTURE")   # offline test hook; unset in Colab


def _guard(args):
    verb = args[0]
    if verb not in _READ_VERBS:
        raise PermissionError(f"gcs(): '{verb}' refused—this notebook is read-only.")
    for a in args[1:]:
        if a.startswith("gs://") and not a.startswith(READ_ONLY_PREFIX):
            raise PermissionError(f"gcs(): {a} is outside {READ_ONLY_PREFIX}")
    GCS_LOG.append("gcloud storage " + " ".join(args))


def _fixture_path(uri):
    return Path(_FIXTURE) / uri[len(GCS_PREFIX):].lstrip("/")


def _fixture_ls(args):
    recursive = "-r" in args
    uri = [a for a in args if a.startswith("gs://")][0]
    root = _fixture_path(uri)
    if not root.exists():
        return subprocess.CompletedProcess(args, 1, b"", b"One or more URLs matched no objects.")
    files = sorted(root.rglob("*") if recursive else root.iterdir())
    lines, total = [], 0
    for f in files:
        if f.is_file():
            rel = f.relative_to(Path(_FIXTURE)).as_posix()
            total += f.stat().st_size
            lines.append(f"{f.stat().st_size:>10}  2026-08-17T00:00:00Z  {GCS_PREFIX}/{rel}")
        elif not recursive:
            lines.append(f"{'':>10}  {GCS_PREFIX}/{f.relative_to(Path(_FIXTURE)).as_posix()}/")
    lines.append(f"TOTAL: {len(lines)} objects, {total} bytes")
    return subprocess.CompletedProcess(args, 0, ("\n".join(lines) + "\n").encode(), b"")


def gcs(*args, binary=False):
    # Read-only gcloud storage. Returns CompletedProcess; stdout is bytes if binary=True.
    _guard(args)
    if _FIXTURE:
        if args[0] == "ls":
            r = _fixture_ls(args)
        else:
            p = _fixture_path(args[-1])
            r = (subprocess.CompletedProcess(args, 0, p.read_bytes(), b"") if p.exists()
                 else subprocess.CompletedProcess(args, 1, b"", b"No URLs matched"))
    else:
        r = subprocess.run(["gcloud", "storage", *args], capture_output=True)
    if not binary:
        r = subprocess.CompletedProcess(r.args, r.returncode,
                                        r.stdout.decode("utf-8", "replace"),
                                        r.stderr.decode("utf-8", "replace"))
    return r


def gcs_stream(uri):
    # Read-only streaming cat. Returns a binary file-like object and a close() callable.
    _guard(("cat", uri))
    if _FIXTURE:
        fh = open(_fixture_path(uri), "rb")
        return fh, fh.close
    p = subprocess.Popen(["gcloud", "storage", "cat", uri],
                         stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    def _close():
        p.stdout.close(); p.kill(); p.wait()
    return p.stdout, _close


# ---- handback collection ----------------------------------------------------------------
HB = {}            # section title -> list of lines; printed together in section 12
TIMERS = {}


def hb(section, line=""):
    HB.setdefault(section, []).append(str(line))


def send_back(section):
    # Print a section's SEND BACK block.
    print("\n" + "─" * 78)
    print(f"📋 SEND BACK—{section}")
    print("─" * 78)
    for l in HB.get(section, []):
        print(l)
    print("─" * 78)


def show(df, index=False, max_rows=60):
    print(df.to_string(index=index, max_rows=max_rows))


def fmt_int(x):
    return f"{int(x):,}" if pd.notna(x) else "—"


def fmt_bytes(n):
    n = float(n)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024 or unit == "TB":
            return f"{n:,.1f} {unit}" if unit != "B" else f"{n:,.0f} B"
        n /= 1024


print("read-only guard armed" + ("  [OFFLINE FIXTURE MODE]" if _FIXTURE else ""))

---
## 2 · Bucket inventory

A plain listing of what is actually under `checkpoints/`, plus the top level of the prefix.
Part 2 mirrors its generation checkpoints (`gen_*.csv`) and embedding checkpoints
(`emb_*.parquet`) to `checkpoints/`. Part 1's eight relational tables were staged one level up
as headerless `*.csv.gz`. Section 3 uses whichever is present for each table.

In [ ]:
_LS_RE = re.compile(r"^\s*(\d+)\s+(\S+)\s+(gs://\S+)\s*$")


def list_objects(uri, recursive=False):
    args = ["ls", "-l"] + (["-r"] if recursive else []) + [uri.rstrip("/") + "/"]
    r = gcs(*args)
    if r.returncode != 0:
        print(f"  ⚠️  could not list {uri}: {r.stderr.strip()[:300]}")
        return pd.DataFrame(columns=["uri", "name", "bytes", "updated"])
    rows = []
    for line in r.stdout.splitlines():
        m = _LS_RE.match(line)
        if m:
            rows.append({"uri": m.group(3), "name": m.group(3).rsplit("/", 1)[-1],
                         "bytes": int(m.group(1)), "updated": m.group(2)})
    return pd.DataFrame(rows)


INV_CKPT = list_objects(CKPT_GCS, recursive=True)
INV_TOP  = list_objects(GCS_PREFIX, recursive=False)
INVENTORY = pd.concat([INV_CKPT, INV_TOP], ignore_index=True).drop_duplicates("uri")
OBJ_BYTES = dict(zip(INVENTORY.uri, INVENTORY.bytes))

S = "2 · Bucket inventory"
print(f"{CKPT_GCS}/   ({len(INV_CKPT)} objects, {fmt_bytes(INV_CKPT.bytes.sum()) if len(INV_CKPT) else '0 B'})\n")
if len(INV_CKPT):
    _v = INV_CKPT.assign(path=INV_CKPT.uri.str.replace(CKPT_GCS + "/", "", regex=False),
                         size=INV_CKPT.bytes.map(fmt_bytes))
    show(_v[["path", "size", "updated"]])
print(f"\n{GCS_PREFIX}/   (top level, {len(INV_TOP)} objects)\n")
if len(INV_TOP):
    _v = INV_TOP.assign(size=INV_TOP.bytes.map(fmt_bytes))
    show(_v[["name", "size", "updated"]])

hb(S, f"checkpoints/: {len(INV_CKPT)} objects, "
      f"{fmt_bytes(INV_CKPT.bytes.sum()) if len(INV_CKPT) else '0 B'}")
for _, r in INV_CKPT.iterrows():
    hb(S, f"  {r.uri.replace(CKPT_GCS + '/', '')}  {fmt_bytes(r.bytes)}")
hb(S, f"top level: {', '.join(INV_TOP.name.tolist()) or '(none listed)'}")
send_back(S)

---
## 3 · Load the tables

The Part 2 loader, made read-only. For each table it uses, in order:

1. a **Parquet copy anywhere under `checkpoints/`** named `<table>.parquet` (Part 1's checkpoint format), or
2. the **staged headerless `<table>.csv.gz`** at the prefix root, with column order from
   `manifest.json` → `staged_files[<table>].column_order`. If no manifest is reachable, the order
   is rebuilt from the staged `schema.sql`, exactly as Part 2 does—but held in memory only.
   Part 2 mirrors a rebuilt manifest back to the bucket; this notebook does not.

Dtypes are forced the Part 2 way: anything id-shaped to nullable `Int64`, dates to datetimes,
text ids to strings. Row counts are checked against Part 2's `EXPECTED_ROWS`.

Then the **scope check** from Part 1 section 2: Big 5 must be present, UEFA competitions are
discovered from the data, and the Lionel Messi canary must pass.

In [ ]:
INT_COLS = {
    "club_id", "player_id", "game_id", "transfer_id", "current_club_id", "player_club_id",
    "home_club_id", "away_club_id", "from_club_id", "to_club_id", "player_in_id",
    "player_assist_id", "season", "last_season", "minute", "attendance", "height_in_cm",
    "squad_size", "foreigners_number", "national_team_players", "stadium_seats", "total_clubs",
    "international_caps", "international_goals", "market_value_in_eur",
    "highest_market_value_in_eur", "net_transfer_record_eur", "transfer_fee",
    "home_club_goals", "away_club_goals", "home_club_position", "away_club_position",
    "yellow_cards", "red_cards", "goals", "assists", "minutes_played",
}
DATE_COLS = {
    "date_of_birth", "contract_expiration_date", "game_date", "appearance_date",
    "event_date", "valuation_date", "transfer_date",
}
TEXT_COLS = {
    "appearance_id", "game_event_id", "competition_id", "domestic_competition_id", "club_code",
    "transfer_season", "round_label", "aggregate", "from_club_name", "to_club_name",
    "player_name", "club_name", "competition_name",
}


def normalize(df):
    for c in df.columns:
        if c in INT_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
        elif c in DATE_COLS:
            df[c] = pd.to_datetime(df[c], errors="coerce")
        elif c in TEXT_COLS:
            df[c] = df[c].astype("string")
    return df


def columns_from_ddl(sql):
    # Part 2's DDL parser: per-table column order from CREATE TABLE statements.
    out = {}
    for m in re.finditer(r"CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?[\"`]?(\w+)[\"`]?\s*\((.*?)\n\s*\)\s*;",
                         sql, re.S | re.I):
        name, body = m.group(1), m.group(2)
        cols, depth = [], 0
        for raw in body.split("\n"):
            line = raw.strip()
            if not line or line.startswith("--"):
                continue
            if depth == 0 and not re.match(
                    r"(?i)(PRIMARY\s+KEY|FOREIGN\s+KEY|CONSTRAINT|UNIQUE|CHECK|EXCLUDE)\b", line):
                tok = line.split()[0].strip('"`,')
                if tok:
                    cols.append(tok)
            depth += line.count("(") - line.count(")")
        if cols:
            out[name] = cols
    return out


def load_manifest():
    for uri in (f"{CKPT_GCS}/manifest.json", f"{GCS_PREFIX}/manifest.json",
                f"{GCS_PREFIX}/artifacts/manifest.json"):
        r = gcs("cat", uri)
        if r.returncode == 0 and r.stdout.strip():
            print(f"manifest read from {uri}")
            return json.loads(r.stdout), uri
    r = gcs("cat", f"{GCS_PREFIX}/schema.sql")
    if r.returncode == 0:
        cols = columns_from_ddl(r.stdout)
        print("no manifest reachable—column order rebuilt from schema.sql (in memory only)")
        return {"staged_files": {t: {"column_order": c} for t, c in cols.items()}}, "schema.sql"
    return {}, None


MANIFEST, MANIFEST_SRC = load_manifest()
PASS2 = {"profile_text", "profile_embedding"}


def find_parquet(table):
    hits = INV_CKPT[INV_CKPT.name == f"{table}.parquet"] if len(INV_CKPT) else INV_CKPT
    return hits.uri.iloc[0] if len(hits) else None


def read_table(table):
    pq = find_parquet(table)
    if pq:
        r = gcs("cat", pq, binary=True)
        if r.returncode == 0:
            return normalize(pd.read_parquet(io.BytesIO(r.stdout))), pq, len(r.stdout)
        print(f"  ⚠️  {pq} listed but unreadable—falling back to the staged CSV")
    entry = (MANIFEST.get("staged_files") or {}).get(table) or {}
    cols = [c for c in entry.get("column_order", []) if c not in PASS2]
    if not cols:
        raise KeyError(f"No Parquet for {table} under checkpoints/ and no column order for its "
                       "headerless CSV. Refusing to guess. List the bucket:\n"
                       f"  gcloud storage ls -r {GCS_PREFIX}/")
    uri = f"{GCS_PREFIX}/{table}.csv.gz"
    r = gcs("cat", uri, binary=True)
    if r.returncode != 0:
        raise FileNotFoundError(f"{uri}: {r.stderr[:300]!r}")
    df = pd.read_csv(io.BytesIO(gzip.decompress(r.stdout)), header=None, names=cols,
                     dtype={c: "string" for c in cols if c in TEXT_COLS}, low_memory=False)
    return normalize(df), uri, len(r.stdout)


t0 = time.time()
T, SRC, SRC_BYTES = {}, {}, {}
for t in TABLES_NEEDED:
    T[t], SRC[t], SRC_BYTES[t] = read_table(t)
    got, want = len(T[t]), EXPECTED_ROWS[t]
    mark = "✅" if got == want else "🔴"
    print(f"  {mark} {t:<14}{got:>10,} rows x {T[t].shape[1]:>2} cols   "
          f"{fmt_bytes(SRC_BYTES[t]):>10}   [{SRC[t].replace(GCS_PREFIX + '/', '')}]")
ROW_MISMATCH = {t: (len(T[t]), EXPECTED_ROWS[t]) for t in T if len(T[t]) != EXPECTED_ROWS[t]}
TIMERS["load"] = time.time() - t0
print(f"\nloaded in {TIMERS['load']:.0f}s")
if ROW_MISMATCH:
    print("🔴 row counts differ from Part 2's EXPECTED_ROWS—this is a different snapshot or a "
          "different scope. Every number below describes THIS data:")
    for t, (g_, w_) in ROW_MISMATCH.items():
        print(f"     {t}: {g_:,} vs expected {w_:,}")

In [ ]:
# ---- Scope check (Part 1, section 2 and the section 4 canary) ----------------------------
S = "3 · Load the tables"
G = T["games"]
SCOPE_COMPS = sorted(G.competition_id.dropna().unique().tolist())
missing_big5 = [c for c in BIG5 if c not in SCOPE_COMPS]
if missing_big5:
    raise RuntimeError(f"Big 5 codes missing from games: {missing_big5}")
UEFA_COMPS = [c for c in SCOPE_COMPS if c not in BIG5]
if not UEFA_COMPS:
    raise RuntimeError("D-15—no UEFA competition in games. The settled scope is Big 5 + UCL/UEL; "
                       "a Big-5-only checkpoint would make every number below wrong. Stopping.")

_cn = T["competitions"].set_index("competition_id").competition_name.to_dict()
COMP_NAME = {c: str(_cn.get(c, c)) for c in SCOPE_COMPS}
CLUB_NAME = T["clubs"].set_index("club_id").club_name.astype(str).to_dict()
P = T["players"]
_pv = P.set_index("player_id")          # name / value lookups used by sections 6–8

lionel =P[P.player_name.str.strip().str.lower() == "lionel messi"]
if len(lionel) != 1:
    raise RuntimeError("CANARY FAILED—Lionel Messi is not present exactly once in players.")

seasons_all = sorted(int(s) for s in G.season.dropna().unique())
print(f"scope competitions ({len(SCOPE_COMPS)}):")
for c in SCOPE_COMPS:
    n = int((G.competition_id == c).sum())
    print(f"   {c:<6} {COMP_NAME[c]:<35} {n:>7,} games")
print(f"\nseasons in games: {seasons_all[0]}–{seasons_all[-1]} ({len(seasons_all)})")
print("✅ canary passes—Lionel Messi present exactly once")

hb(S, f"sources: " + "; ".join(f"{t}={SRC[t].replace(GCS_PREFIX + '/', '')}" for t in TABLES_NEEDED))
hb(S, f"manifest: {MANIFEST_SRC}")
hb(S, "rows: " + ", ".join(f"{t} {len(T[t]):,}" for t in TABLES_NEEDED))
hb(S, "row-count check vs EXPECTED_ROWS: " + ("all match" if not ROW_MISMATCH else
      "MISMATCH " + ", ".join(f"{t} {g_:,}≠{w_:,}" for t, (g_, w_) in ROW_MISMATCH.items())))
hb(S, f"scope: {', '.join(SCOPE_COMPS)}  (UEFA discovered: {', '.join(UEFA_COMPS)})")
hb(S, f"seasons: {seasons_all[0]}–{seasons_all[-1]}   load time {TIMERS['load']:.0f}s")
send_back(S)

---
## 4 · Player-seasons in scope

A **player-season** is one player with at least one appearance in a scope competition (Big 5
league, Champions League, Europa League) in one season. `season` is the starting year, so 2024/25
is 2024. Minutes are summed across all scope competitions that season.

This is the unit a per-season scouting report would hang on, so it sets the corpus ceiling at one
report per player-season. The outline promised 150K–250K reports.

**Partial seasons**—a season with fewer than 60% of the median number of games—are flagged.
The snapshot is dated 2026-08-14, so the newest season will be partial.

In [ ]:
S = "4 · Player-seasons in scope"
t0 = time.time()

g = G[["game_id", "competition_id", "season", "game_date", "home_club_id", "away_club_id"]] \
      .rename(columns={"competition_id": "game_comp"})
A = T["appearances"][["game_id", "player_id", "player_club_id", "minutes_played", "goals"]] \
      .merge(g, on="game_id", how="inner")
if len(A) != len(T["appearances"]):
    print(f"⚠️  {len(T['appearances']) - len(A):,} appearances have no matching game—excluded")
A["season"] = A["season"].astype(int)
A["minutes_played"] = A["minutes_played"].fillna(0).astype(int)
A["is_league"] = A.game_comp.isin(BIG5)
A["is_euro"] = A.game_comp.isin(UEFA_COMPS)

PS = (A.groupby(["player_id", "season"])
        .agg(apps=("game_id", "size"), minutes=("minutes_played", "sum"),
             league_apps=("is_league", "sum"), euro_apps=("is_euro", "sum"),
             n_clubs=("player_club_id", "nunique"),
             first_app=("game_date", "min"), last_app=("game_date", "max"))
        .reset_index())


def _mode_comp(mask):
    x = (A[mask].groupby(["player_id", "season", "game_comp"]).size().reset_index(name="n")
           .sort_values(["n", "game_comp"], ascending=[False, True])
           .drop_duplicates(["player_id", "season"]))
    return x[["player_id", "season", "game_comp"]]


PS = PS.merge(_mode_comp(A.is_league).rename(columns={"game_comp": "league_comp"}),
              on=["player_id", "season"], how="left")
PS = PS.merge(_mode_comp(A.is_euro).rename(columns={"game_comp": "euro_comp"}),
              on=["player_id", "season"], how="left")
PS["primary_comp"] = PS.league_comp.fillna(PS.euro_comp)

games_per_season = g.groupby("season").size()
games_per_season.index = games_per_season.index.astype(int)
_median = games_per_season.median()
PARTIAL_SEASONS = sorted(int(s) for s, n in games_per_season.items()
                         if n < PARTIAL_SEASON_SHARE * _median)
COMPLETE_SEASONS = [s for s in seasons_all if s not in PARTIAL_SEASONS]
LATEST_COMPLETE = max(COMPLETE_SEASONS)

N_PS = len(PS)
TH = {t: int((PS.minutes >= t).sum()) for t in MINUTE_THRESHOLDS}

# ---- by season -------------------------------------------------------------------------
by_season = pd.DataFrame({"games": games_per_season})
by_season["player_seasons"] = PS.groupby("season").size()
for t in MINUTE_THRESHOLDS[1:]:
    by_season[f">={t} min"] = PS[PS.minutes >= t].groupby("season").size()
by_season = by_season.fillna(0).astype(int)
by_season["note"] = ["partial" if s in PARTIAL_SEASONS else "" for s in by_season.index]
by_season.index.name = "season"

# ---- by competition --------------------------------------------------------------------
comp_ps = A.drop_duplicates(["player_id", "season", "game_comp"]) \
           .merge(PS[["player_id", "season", "minutes"]], on=["player_id", "season"])
by_comp = pd.DataFrame({
    "competition": [COMP_NAME[c] for c in SCOPE_COMPS],
    "player_seasons_with_an_app": [int((comp_ps.game_comp == c).sum()) for c in SCOPE_COMPS],
    ">=900 min (season total)": [int(((comp_ps.game_comp == c) & (comp_ps.minutes >= 900)).sum())
                                 for c in SCOPE_COMPS],
    "primary_comp (sums to total)": [int((PS.primary_comp == c).sum()) for c in SCOPE_COMPS],
}, index=SCOPE_COMPS)
euro_only = int((PS.league_apps == 0).sum())

TIMERS["player_seasons"] = time.time() - t0

print(f"PLAYER-SEASONS IN SCOPE: {N_PS:,}   ({PS.player_id.nunique():,} distinct players, "
      f"{N_PS / PS.player_id.nunique():.1f} seasons per player on average)\n")
print("At minutes-played thresholds (season total across scope competitions):")
for t in MINUTE_THRESHOLDS:
    print(f"   >= {t:>4} min   {TH[t]:>9,}   ({TH[t] / N_PS:.0%})")
print("\nBy season:")
show(by_season, index=True)
print("\nBy competition (the first two columns overlap—a player-season can appear in a league "
      "AND a UEFA competition; 'primary_comp' assigns each player-season once, league first):")
show(by_comp, index=True)
print(f"\nplayer-seasons with UEFA apps but no Big 5 league apps: {euro_only:,} "
      f"(clubs from outside the Big 5 that met Big 5 sides in Europe)")

lo, hi = OUTLINE_PROMISE
verdict_bits = []
for t in MINUTE_THRESHOLDS:
    verdict_bits.append(f">={t}: {TH[t]:,} = {TH[t] / lo:.0%} of the {lo:,} floor")
print("\nAGAINST THE OUTLINE (one report per player-season):")
for v in verdict_bits:
    print("   " + v)
in_suspect = SUSPECTED_RANGE[0] <= N_PS <= SUSPECTED_RANGE[1]
print(f"   total {N_PS:,}—{'inside' if in_suspect else 'outside'} the suspected "
      f"{SUSPECTED_RANGE[0]:,}–{SUSPECTED_RANGE[1]:,} range")

hb(S, f"player-seasons total: {N_PS:,}  ({PS.player_id.nunique():,} players)")
hb(S, "thresholds: " + "  ".join(f">={t}min {TH[t]:,}" for t in MINUTE_THRESHOLDS))
hb(S, f"partial seasons: {PARTIAL_SEASONS or 'none'}   latest complete: {LATEST_COMPLETE}")
hb(S, "by season (all / >=900):  " + "  ".join(
    f"{s}:{r.player_seasons:,}/{r['>=900 min']:,}" for s, r in by_season.iterrows()))
hb(S, "by competition (with an app / primary):  " + "  ".join(
    f"{c}:{r.player_seasons_with_an_app:,}/{r['primary_comp (sums to total)']:,}"
    for c, r in by_comp.iterrows()))
hb(S, f"UEFA-only player-seasons: {euro_only:,}")
hb(S, f"vs outline floor {lo:,}: " + "; ".join(verdict_bits))
send_back(S)

---
## 5 · Scouting perspectives—2 or 3 clubs per player-season

The multiplier idea: instead of one report per player-season, two or three *clubs* each scout
the player that season. Three perspectives are tested, each from data we hold:

| Perspective | Available when | Club used |
| :-- | :-- | :-- |
| **Later signer** | a transfer **after his first appearance that season** took him to an in-scope club he didn't play for that season | that club (any time later; a ≤2-summer horizon is reported too) |
| **Same league** | he played in a Big 5 league that season | another club from that league and season. For UEFA-only player-seasons, another club with the same `domestic_competition_id` in `clubs` |
| **European opponent** | he played a UCL/UEL match that season | a club he faced |

The perspectives must be **distinct clubs**: the later signer is picked first, preferring one he
didn't also face in Europe, then the European opponent, then a same-league club.

Three counts for each scouts-per-player-season setting *k*:

- **capped**—every player-season gets `min(k, perspectives available)` reports. The realistic count.
- **strict**—only player-seasons with all *k* perspectives get any (k each).
- **ideal**—k × player-seasons. Reachable only by letting two same-league clubs both scout him,
  which drops the "different viewpoint" argument.

`ps_clubs` is held as sets per player-season, so this cell takes a little longer than the others.

In [ ]:
S = "5 · Scouting perspectives"
t0 = time.time()
KEY = ["player_id", "season"]


def _setagg(df, col):
    d = df.dropna(subset=[col])
    return d.groupby(KEY)[col].agg(lambda s: frozenset(int(x) for x in s.unique())).to_dict()


ps_clubs = _setagg(A, "player_club_id")

# European opponents
eu = A[A.is_euro].copy()
_home = eu.player_club_id.eq(eu.home_club_id).fillna(False).astype(bool)
eu["opp"] = eu.away_club_id.where(_home, eu.home_club_id)
euro_opps = _setagg(eu, "opp")

# League clubs per (league, season)
_lg = g[g.game_comp.isin(BIG5)]
_lc = pd.concat([_lg[["game_comp", "season", "home_club_id"]].rename(columns={"home_club_id": "club"}),
                 _lg[["game_comp", "season", "away_club_id"]].rename(columns={"away_club_id": "club"})])
_lc = _lc.dropna(subset=["club"])
_lc["season"] = _lc["season"].astype(int)
league_clubs = _lc.groupby(["game_comp", "season"]).club \
                  .agg(lambda s: frozenset(int(x) for x in s.unique())).to_dict()

# Fallback "same league" for UEFA-only player-seasons: clubs sharing a domestic_competition_id
_cl = T["clubs"].dropna(subset=["domestic_competition_id"])
club_dom = dict(zip(_cl.club_id.astype(int), _cl.domestic_competition_id.astype(str)))
clubs_by_dom = _cl.groupby("domestic_competition_id").club_id \
                  .agg(lambda s: frozenset(int(x) for x in s)).to_dict()

# Later signers
_tr = T["transfers"][["player_id", "transfer_date", "to_club_id"]].dropna()
_m = PS[["player_id", "season", "first_app"]].merge(_tr, on="player_id")
_m = _m[_m.transfer_date > _m.first_app].copy()
_m["horizon_end"] = pd.to_datetime((_m.season + 1 + LATER_SIGNER_HORIZON_YEARS).astype(str) + "-07-01")
later_any = _setagg(_m, "to_club_id")
later_h = _setagg(_m[_m.transfer_date < _m.horizon_end], "to_club_id")

recs = []
for r in PS[["player_id", "season", "league_comp"]].itertuples(index=False):
    key = (r.player_id, r.season)
    own = ps_clubs.get(key, frozenset())
    L = set(later_any.get(key, ())) - own
    Lh = set(later_h.get(key, ())) - own
    E = set(euro_opps.get(key, ())) - own
    if isinstance(r.league_comp, str):
        Sx, s_src = set(league_clubs.get((r.league_comp, r.season), ())) - own, "big5"
    else:
        Sx = set()
        for c in own:
            Sx |= set(clubs_by_dom.get(club_dom.get(c), ()))
        Sx -= own
        s_src = "dom" if Sx else "none"
    picked, n = set(), 0
    if L:
        p = min(L - E) if (L - E) else min(L)
        picked.add(p); n += 1
    E2 = E - picked
    if E2:
        picked.add(min(E2)); n += 1
    if Sx - picked:
        n += 1
    recs.append((bool(L), bool(Lh), bool(E), bool(Sx), s_src, n))

PSP = pd.DataFrame(recs, columns=["has_later", "has_later_h", "has_euro", "has_league",
                                  "league_src", "n_persp"])
PS = pd.concat([PS.reset_index(drop=True), PSP], axis=1)
TIMERS["perspectives"] = time.time() - t0


def report_counts(df):
    out = {}
    for k in SCOUTS_PER_PS:
        out[f"k={k} capped"] = int(np.minimum(df.n_persp, k).sum())
        out[f"k={k} strict"] = int(k * (df.n_persp >= k).sum())
        out[f"k={k} ideal"] = int(k * len(df))
    return out


rc_rows = []
for t in MINUTE_THRESHOLDS:
    d = PS[PS.minutes >= t]
    rc_rows.append({"min minutes": t, "player_seasons": len(d), **report_counts(d)})
RC = pd.DataFrame(rc_rows)

avail = pd.Series({
    "later signer (any time)": PS.has_later.mean(),
    f"later signer (<= {LATER_SIGNER_HORIZON_YEARS} summers)": PS.has_later_h.mean(),
    "same league": PS.has_league.mean(),
    "   of which via clubs.domestic_competition_id": (PS.league_src == "dom").mean(),
    "European opponent": PS.has_euro.mean(),
})
dist = PS.n_persp.value_counts().sort_index()

print("PERSPECTIVE AVAILABILITY (share of all player-seasons)")
for k, v in avail.items():
    print(f"   {k:<48} {v:>6.1%}")
print("\nDistinct perspectives available per player-season:")
for k, v in dist.items():
    print(f"   {k}: {v:>9,}  ({v / N_PS:.1%})")
print("\nREPORT COUNTS by minutes threshold")
show(RC)

_c900 = RC.set_index("min minutes").loc[900]
_c1 = RC.set_index("min minutes").loc[1]
hb(S, "availability: " + "  ".join(f"{k.strip()} {v:.0%}" for k, v in avail.items()))
hb(S, "n_persp distribution: " + "  ".join(f"{k}:{v:,}" for k, v in dist.items()))
for t in MINUTE_THRESHOLDS:
    r = RC.set_index("min minutes").loc[t]
    hb(S, f">={t}min ({int(r.player_seasons):,} ps): " + "  ".join(
        f"k={k} capped {int(r[f'k={k} capped']):,} / strict {int(r[f'k={k} strict']):,}"
        for k in SCOUTS_PER_PS[1:]))
send_back(S)

---
## 6 · Extended absences

An **extended absence** is a run of consecutive club matches the player missed, *between two of
his own appearances* for that club in the same season, while the club kept playing. Those
become "extended absence" facts in a scouting report.

How it's measured:

- Each club's schedule per season is its scope matches (Big 5 league + UCL/UEL) in date order.
- A player's stint is (player, season, club). A mid-season transfer starts a new stint, so moving
  clubs never reads as an absence.
- The stint needs at least `ABSENCE_MIN_APPS` appearances, so a fringe player's normal
  in-and-out pattern isn't counted.
- The headline definition is `ABSENCE_PRIMARY`: **≥ 6 consecutive club matches and ≥ 30 days.**

**What the data cannot say.** `appearances` only records matches he played. Injury, suspension,
being dropped, and international tournaments all look identical. Domestic cups are out of scope,
so "the club kept playing" is measured on league and European matches only.

**Season-ending absences**—he played up to some point, the club played on, he never returned —
are counted separately, because they're a different fact ("missed the run-in"). Partial seasons
are excluded from that count.

In [ ]:
S = "6 · Extended absences"
t0 = time.time()

cg = pd.concat([g[["game_id", "season", "game_date", "home_club_id"]].rename(columns={"home_club_id": "club_id"}),
                g[["game_id", "season", "game_date", "away_club_id"]].rename(columns={"away_club_id": "club_id"})])
cg = cg.dropna(subset=["club_id"]).sort_values(["club_id", "season", "game_date", "game_id"])
cg["season"] = cg["season"].astype(int)
cg["idx"] = cg.groupby(["club_id", "season"]).cumcount()
club_last_idx = cg.groupby(["club_id", "season"]).idx.max()

SEG = ["player_id", "season", "player_club_id"]
a = A[["player_id", "season", "player_club_id", "game_id", "game_date"]].dropna(subset=["player_club_id"]) \
      .merge(cg[["game_id", "club_id", "idx"]], left_on=["game_id", "player_club_id"],
             right_on=["game_id", "club_id"], how="left")
unmatched = int(a.idx.isna().sum())
a = a.dropna(subset=["idx"]).copy()
a["idx"] = a["idx"].astype(int)
a = a.sort_values(SEG + ["idx"])
grp = a.groupby(SEG)
a["seg_apps"] = grp.idx.transform("size")
a["gap_games"] = grp.idx.diff() - 1
a["gap_days"] = grp.game_date.diff().dt.days
a["gap_from"] = grp.game_date.shift()

GAPS = a[(a.seg_apps >= ABSENCE_MIN_APPS) & (a.gap_games >= 1)].copy()
g_prim, d_prim = ABSENCE_PRIMARY
GAPS["primary"] = (GAPS.gap_games >= g_prim) & (GAPS.gap_days >= d_prim)

rows = []
for n in ABSENCE_GAP_GAMES:
    sel = GAPS[GAPS.gap_games >= n]
    rows.append({"definition": f">= {n} matches", "absence facts": len(sel),
                 "player-seasons": sel[KEY].drop_duplicates().shape[0]})
for dd in ABSENCE_GAP_DAYS:
    sel = GAPS[GAPS.gap_days >= dd]
    rows.append({"definition": f">= {dd} days", "absence facts": len(sel),
                 "player-seasons": sel[KEY].drop_duplicates().shape[0]})
sel = GAPS[GAPS.primary]
rows.append({"definition": f"PRIMARY: >= {g_prim} matches AND >= {d_prim} days",
             "absence facts": len(sel), "player-seasons": sel[KEY].drop_duplicates().shape[0]})
ABS = pd.DataFrame(rows)
N_ABS_PRIMARY = int(GAPS.primary.sum())
N_ABS_PS = int(GAPS[GAPS.primary][KEY].drop_duplicates().shape[0])

# Share of player-seasons eligible (>= ABSENCE_MIN_APPS in some stint) that have a primary gap
eligible_ps = a[a.seg_apps >= ABSENCE_MIN_APPS][KEY].drop_duplicates().shape[0]

# Season-ending absences
seg_end = a.groupby(SEG).agg(last_idx=("idx", "max"), last_date=("game_date", "max"),
                             seg_apps=("idx", "size")).reset_index()
ps_last = a.groupby(KEY).game_date.max().rename("ps_last")
seg_end = seg_end.merge(ps_last, on=KEY)
seg_end = seg_end[(seg_end.last_date == seg_end.ps_last) & (seg_end.seg_apps >= ABSENCE_MIN_APPS)
                  & (~seg_end.season.isin(PARTIAL_SEASONS))]
seg_end["club_last"] = [club_last_idx.get((int(c), int(s)), np.nan)
                        for c, s in zip(seg_end.player_club_id, seg_end.season)]
seg_end["trailing"] = seg_end.club_last - seg_end.last_idx
TRAIL = {n: int((seg_end.trailing >= n).sum()) for n in ABSENCE_GAP_GAMES}

by_season_abs = GAPS[GAPS.primary].groupby("season").size().reindex(seasons_all, fill_value=0)
buckets = pd.cut(GAPS.gap_games, [0, 1, 2, 3, 5, 9, 14, 1000],
                 labels=["1", "2", "3", "4-5", "6-9", "10-14", "15+"]).value_counts().sort_index()

# Examples: primary gaps for the most valuable players
ex =GAPS[GAPS.primary].copy()
ex["value"] = ex.player_id.map(_pv.highest_market_value_in_eur)
ex = ex.sort_values(["value", "gap_games"], ascending=False).drop_duplicates("player_id").head(10)
ex_view = pd.DataFrame({
    "player": ex.player_id.map(_pv.player_name),
    "season": ex.season.map(lambda s: f"{s}/{str(s + 1)[-2:]}"),
    "club": ex.player_club_id.map(lambda c: CLUB_NAME.get(int(c), c)),
    "last app": ex.gap_from.dt.date, "back": ex.game_date.dt.date,
    "matches missed": ex.gap_games.astype(int), "days": ex.gap_days.astype(int),
})
TIMERS["absences"] = time.time() - t0

if unmatched:
    print(f"⚠️  {unmatched:,} appearances where player_club_id was neither home nor away—skipped")
print(f"stints with >= {ABSENCE_MIN_APPS} apps cover {eligible_ps:,} player-seasons\n")
print("MID-SEASON ABSENCES (between two appearances, club kept playing)")
show(ABS)
print(f"\nheadline: {N_ABS_PRIMARY:,} absence facts across {N_ABS_PS:,} player-seasons "
      f"({N_ABS_PS / max(eligible_ps, 1):.1%} of eligible player-seasons)")
print("\nGap length distribution (consecutive club matches missed, all gaps >= 1):")
for k, v in buckets.items():
    print(f"   {k:>6}: {v:>8,}")
print("\nPrimary absences by season:")
print("   " + "  ".join(f"{s}:{v:,}" for s, v in by_season_abs.items()))
print("\nSEASON-ENDING absences (never returned that season; partial seasons excluded)—"
      "separate from the counts above:")
for n, v in TRAIL.items():
    print(f"   >= {n:>2} club matches after his last app: {v:>7,}")
print("\nExamples—primary absences for the highest-valued players:")
show(ex_view)

hb(S, f"eligible player-seasons (stint >= {ABSENCE_MIN_APPS} apps): {eligible_ps:,}")
for _, r in ABS.iterrows():
    hb(S, f"{r.definition}: {r['absence facts']:,} facts / {r['player-seasons']:,} player-seasons")
hb(S, "season-ending: " + "  ".join(f">={n}: {v:,}" for n, v in TRAIL.items()))
hb(S, "primary by season: " + "  ".join(f"{s}:{v:,}" for s, v in by_season_abs.items()))
hb(S, "examples: " + "; ".join(f"{r.player} {r.season} {r.club} {r['matches missed']}m/{r.days}d"
                                for _, r in ex_view.head(5).iterrows()))
send_back(S)

---
## 7 · Transfer graph

`transfers` holds every recorded move for the 13,439 in-scope players—their whole careers, so
it reaches well outside the scope seasons and leagues. Club ids are NULL wherever the club isn't
in the 796-row `clubs` table (S-24), so the graph is built on **club names**, which are always
populated. Ids only decide whether a club is "in scope".

- **Non-club states** ("Without Club", "Unknown", "Retired"…) are not nodes.
- **Youth and reserve sides** (U19, U21, B, II, Castilla…) are kept in the move counts but
  dropped from chains, so "Ajax U21 → Ajax" doesn't count as a hop.
- A **chain** is a run of consecutive moves where each move starts at the club the previous one
  ended at: X → Y → Z. Loan returns show up as X → Y → X, so chains are also counted by
  **distinct** clubs.
- `transfer_season` is derived from `transfer_date`: a July–June window starting in that year.

In [ ]:
S = "7 · Transfer graph"
t0 = time.time()
TR = T["transfers"].copy()
_y = TR.transfer_date.dt.year
TR["tseason"] = _y.where(TR.transfer_date.dt.month >= 7, _y - 1).astype("Int64")
for side in ("from", "to"):
    nm = TR[f"{side}_club_name"].fillna("").astype(str)
    TR[f"{side}_nonclub"] = nm.str.match(NON_CLUB_RE, case=False) | (nm.str.strip() == "")
    TR[f"{side}_reserve"] = nm.str.contains(RESERVE_RE, case=False, regex=True)
    TR[f"{side}_real"] = ~TR[f"{side}_nonclub"]
TR["c2c"] = TR.from_real & TR.to_real & (TR.from_club_name != TR.to_club_name)
TR["both_in_scope"] = TR.from_club_id.notna() & TR.to_club_id.notna()
TR["fee_known_paid"] = TR.transfer_fee.fillna(-1) > 0
TR["free"] = TR.transfer_fee.fillna(-1) == 0
TR["fee_unknown"] = TR.transfer_fee.isna()

first_scope = seasons_all[0]
per = (TR[TR.tseason >= first_scope - 2].groupby("tseason")
         .agg(moves=("transfer_id", "size"), club_to_club=("c2c", "sum"),
              both_in_scope=("both_in_scope", "sum"), paid_fee=("fee_known_paid", "sum"),
              free=("free", "sum"), fee_unknown=("fee_unknown", "sum"))
         .astype(int))
per.index = [f"{s}/{str(s + 1)[-2:]}" for s in per.index]
earlier = int((TR.tseason < first_scope - 2).sum())

# Busiest clubs (by name), with fee totals
outs = TR[TR.from_real].groupby("from_club_name").agg(out_moves=("transfer_id", "size"),
                                                      received_eur=("transfer_fee", "sum"))
ins = TR[TR.to_real].groupby("to_club_name").agg(in_moves=("transfer_id", "size"),
                                                 spent_eur=("transfer_fee", "sum"))
busy = ins.join(outs, how="outer").fillna(0)
busy["total_moves"] = busy.in_moves + busy.out_moves
_inscope_names = set(TR.loc[TR.to_club_id.notna(), "to_club_name"]) | \
                 set(TR.loc[TR.from_club_id.notna(), "from_club_name"])
busy["in_scope"] = ["yes" if n in _inscope_names else "" for n in busy.index]
busy = busy.sort_values("total_moves", ascending=False)
busy.index.name = "club"
busy_view = busy.head(20).assign(
    spent=lambda d: (d.spent_eur / 1e6).round(0).astype(int).map(lambda x: f"€{x:,}m"),
    received=lambda d: (d.received_eur / 1e6).round(0).astype(int).map(lambda x: f"€{x:,}m"),
)[["in_moves", "out_moves", "total_moves", "spent", "received", "in_scope"]].astype(
    {"in_moves": int, "out_moves": int, "total_moves": int})

# Graph shape
c2c = TR[TR.c2c]
nodes = pd.unique(pd.concat([c2c.from_club_name, c2c.to_club_name]))
edges = c2c.groupby(["from_club_name", "to_club_name"]).size().sort_values(ascending=False)
deg = pd.concat([c2c.from_club_name, c2c.to_club_name]).value_counts()

# Chains
chains = []
_cols = ["player_id", "transfer_date", "from_club_name", "to_club_name", "from_club_id",
         "to_club_id", "transfer_fee", "from_real", "to_real", "from_reserve", "to_reserve"]
for pid, grp_ in TR.sort_values(["player_id", "transfer_date", "transfer_id"])[_cols].groupby("player_id", sort=False):
    cur = None
    for r in grp_.itertuples(index=False):
        ok = r.from_real and r.to_real and r.from_club_name != r.to_club_name
        if EXCLUDE_RESERVE_SIDES_FROM_CHAINS:
            ok = ok and not r.from_reserve and not r.to_reserve
        if not ok:
            if cur and len(cur["clubs"]) >= 3:
                chains.append(cur)
            cur = None
            continue
        if cur and cur["clubs"][-1][0] == r.from_club_name:
            cur["clubs"].append((r.to_club_name, r.to_club_id, r.transfer_date, r.transfer_fee))
        else:
            if cur and len(cur["clubs"]) >= 3:
                chains.append(cur)
            cur = {"player_id": pid,
                   "clubs": [(r.from_club_name, r.from_club_id, None, None),
                             (r.to_club_name, r.to_club_id, r.transfer_date, r.transfer_fee)]}
    if cur and len(cur["clubs"]) >= 3:
        chains.append(cur)

CH = pd.DataFrame([{
    "player_id": c["player_id"], "hops": len(c["clubs"]) - 1,
    "distinct": len({x[0] for x in c["clubs"]}),
    "in_scope_clubs": len({x[0] for x in c["clubs"] if pd.notna(x[1])}),
    "chain": c["clubs"],
} for c in chains])
if len(CH):
    CH["value"] = CH.player_id.map(_pv.highest_market_value_in_eur).fillna(0)


def _fmt_chain(cl):
    parts = [cl[0][0]]
    for name, _id, d, fee in cl[1:]:
        tag = []
        if d is not None and pd.notna(d):
            tag.append(str(pd.Timestamp(d).year))
        if fee is not None and pd.notna(fee) and fee > 0:
            tag.append(f"€{fee / 1e6:,.1f}m")
        elif fee is not None and pd.notna(fee) and fee == 0:
            tag.append("free")
        parts.append(f"{name}{' (' + ', '.join(tag) + ')' if tag else ''}")
    return " → ".join(parts)


n2 = int((CH.hops >= 2).sum()) if len(CH) else 0
n3 = int((CH.hops >= 3).sum()) if len(CH) else 0
nd3 = int((CH.distinct >= 3).sum()) if len(CH) else 0
nsc3 = int((CH.in_scope_clubs >= 3).sum()) if len(CH) else 0
players_with = CH[CH.distinct >= 3].player_id.nunique() if len(CH) else 0
examples = (CH[(CH.distinct >= 3) & (CH.in_scope_clubs >= 3)]
              .sort_values(["value", "in_scope_clubs"], ascending=False)
              .drop_duplicates("player_id").head(CHAIN_EXAMPLES)) if len(CH) else CH
moves_per_player = TR.groupby("player_id").size()
TIMERS["transfers"] = time.time() - t0

print(f"transfers: {len(TR):,} moves for {TR.player_id.nunique():,} players "
      f"(median {moves_per_player.median():.0f}, max {moves_per_player.max()} per player)")
print(f"club-to-club {int(TR.c2c.sum()):,}   both ends in scope {int(TR.both_in_scope.sum()):,}   "
      f"paid fee {int(TR.fee_known_paid.sum()):,}   free {int(TR.free.sum()):,}   "
      f"fee unknown {int(TR.fee_unknown.sum()):,}\n")
print(f"MOVES PER SEASON (from {first_scope - 2}/{str(first_scope - 1)[-2:]}; {earlier:,} earlier moves not shown)")
show(per, index=True)
print("\nBUSIEST CLUBS (by name; in_scope = has a row in clubs)")
show(busy_view, index=True)
print(f"\nGRAPH: {len(nodes):,} club nodes, {len(edges):,} distinct directed routes, "
      f"{len(c2c):,} club-to-club moves; median degree {deg.median():.0f}, max {deg.max():,}")
print("Top routes:")
for (f, t_), v in edges.head(10).items():
    print(f"   {v:>4}  {f} → {t_}")
print(f"\nCHAINS (reserve sides {'excluded' if EXCLUDE_RESERVE_SIDES_FROM_CHAINS else 'included'}):")
print(f"   >= 2 hops (X→Y→Z): {n2:,}   >= 3 hops: {n3:,}")
print(f"   >= 3 distinct clubs: {nd3:,} chains, {players_with:,} players")
print(f"   >= 3 distinct IN-SCOPE clubs: {nsc3:,} chains")
print("\nExamples (highest-valued players with a 3+ in-scope-club chain):")
for _, r in examples.iterrows():
    print(f"   {_pv.player_name.get(r.player_id, r.player_id)}: {_fmt_chain(r.chain)}")

hb(S, f"moves {len(TR):,}; club-to-club {int(TR.c2c.sum()):,}; both in scope {int(TR.both_in_scope.sum()):,}; "
      f"paid {int(TR.fee_known_paid.sum()):,}; free {int(TR.free.sum()):,}; fee unknown {int(TR.fee_unknown.sum()):,}")
hb(S, "per season (moves/both-in-scope): " + "  ".join(f"{s}:{r.moves:,}/{r.both_in_scope:,}"
                                                     for s, r in per.iterrows()))
hb(S, "busiest: " + "; ".join(f"{n} {int(r.total_moves)}" for n, r in busy.head(10).iterrows()))
hb(S, f"graph: {len(nodes):,} nodes, {len(edges):,} routes, max degree {deg.max():,}")
hb(S, f"chains: >=2 hops {n2:,}; >=3 hops {n3:,}; >=3 distinct {nd3:,} ({players_with:,} players); "
      f">=3 in-scope {nsc3:,}")
for _, r in examples.head(5).iterrows():
    hb(S, f"  e.g. {_pv.player_name.get(r.player_id, r.player_id)}: {_fmt_chain(r.chain)}")
send_back(S)

---
## 8 · Players who share a name

Part 1 already warns that names aren't unique (`players.player_name` carries a column comment
about the Paulinhos). For scouting reports the harder case is two same-named players **active in
the same season**, because then season alone doesn't tell them apart.

Counted three ways: exact name, accent- and case-folded name (so "Vitinha" and "Vitínha" collide),
and same name in the same season.

In [ ]:
S = "8 · Players who share a name"
P2 = P[["player_id", "player_name", "date_of_birth", "main_position", "detailed_position",
        "country_of_citizenship", "highest_market_value_in_eur"]].copy()
P2["name"] = P2.player_name.astype(str).str.strip()
P2["folded"] = P2.name.map(lambda s: "".join(ch for ch in unicodedata.normalize("NFKD", s)
                                              if not unicodedata.combining(ch)).lower())
cnt = P2.groupby("name").player_id.nunique()
shared = cnt[cnt >= 2].sort_values(ascending=False)
fcnt = P2.groupby("folded").player_id.nunique()
fshared = fcnt[fcnt >= 2]

ids_shared = set(P2[P2.name.isin(shared.index)].player_id)
ids_fshared = set(P2[P2.folded.isin(fshared.index)].player_id)
ps_shared = PS[PS.player_id.isin(ids_shared)]
psn = PS.merge(P2[["player_id", "name"]], on="player_id")
same_season = psn.groupby(["name", "season"]).player_id.transform("nunique") >= 2
n_same_season_ps = int(same_season.sum())
n_same_season_names = psn[same_season].name.nunique()

print(f"EXACT names shared by 2+ players: {len(shared):,} names, {len(ids_shared):,} players")
print(f"   covering {len(ps_shared):,} player-seasons ({len(ps_shared) / N_PS:.1%}); "
      f"{int((ps_shared.minutes >= 900).sum()):,} of them at >= 900 min")
print(f"FOLDED (accent/case-insensitive): {len(fshared):,} names, {len(ids_fshared):,} players")
print(f"SAME NAME, SAME SEASON: {n_same_season_ps:,} player-seasons under {n_same_season_names:,} names "
      "— season alone cannot disambiguate these")
print("\nMost-shared names:")
for nm, v in shared.head(15).items():
    print(f"   {v}  {nm}")

# The Paulinhos
pau = P2[P2.folded == "paulinho"].copy()
_apps_p = A[A.player_id.isin(pau.player_id)]
_clubs_p = _apps_p.dropna(subset=["player_club_id"]).groupby("player_id").player_club_id.agg(
    lambda s: ", ".join(sorted({CLUB_NAME.get(int(c), str(c)) for c in s.unique()})))
_ps_p = PS[PS.player_id.isin(pau.player_id)].groupby("player_id").agg(
    player_seasons=("season", "size"), seasons=("season", lambda s: f"{s.min()}–{s.max()}"),
    minutes=("minutes", "sum"))
pau_view = pau.set_index("player_id").join(_ps_p).assign(clubs=_clubs_p)
pau_view = pau_view.reset_index()[["player_id", "name", "date_of_birth", "detailed_position",
                                   "country_of_citizenship", "player_seasons", "seasons",
                                   "minutes", "clubs"]]
pau_view["date_of_birth"] = pau_view.date_of_birth.dt.date
print(f"\nTHE PAULINHOS ({len(pau)} players, {int(pau_view.player_seasons.fillna(0).sum()):,} player-seasons):")
show(pau_view)

hb(S, f"exact shared names: {len(shared):,} names / {len(ids_shared):,} players / "
      f"{len(ps_shared):,} player-seasons ({len(ps_shared) / N_PS:.1%}); >=900min {int((ps_shared.minutes >= 900).sum()):,}")
hb(S, f"folded shared names: {len(fshared):,} names / {len(ids_fshared):,} players")
hb(S, f"same name same season: {n_same_season_ps:,} player-seasons / {n_same_season_names:,} names")
hb(S, "top names: " + ", ".join(f"{nm} ({v})" for nm, v in shared.head(10).items()))
hb(S, f"Paulinhos: {len(pau)} players, {int(pau_view.player_seasons.fillna(0).sum()):,} player-seasons: " +
      "; ".join(f"{int(r.player_id)} b.{r.date_of_birth} {r.seasons} {str(r.clubs)[:40]}" for _, r in pau_view.iterrows()))
send_back(S)

---
## 9 · Size math at 3072-dim float32

### What gets measured first (all read-only)

- **Text bytes per word**, from the existing Lab 1 profile text in Part 2's generation checkpoint
  (`checkpoints/gen_players_*.csv`). This replaces `DEFAULT_BYTES_PER_WORD`.
- **CSV bytes per vector row and gzip ratio**, from the first few hundred records of the staged
  `players_profiles.csv.gz` (streamed, never downloaded whole) and its object size. That's what
  sets staging size and COPY wire bytes.

### The per-row model

One report row = metadata + 120–180 words of text + one `vector(3072)`:

- **Vector**—pgvector stores 4 bytes per dimension plus an 8-byte header: **12,296 bytes**.
  That's far over the ~2 KB TOAST threshold, so it lives out of line in 1,996-byte TOAST chunks,
  each with its own tuple overhead and an index entry. Float32 barely compresses, so no
  compression is assumed.
- **Heap tuple**—tuple header, metadata, inline text (≈ 1 KB at these lengths, under the
  TOAST threshold), and an 18-byte TOAST pointer.
- **ScaNN** (`sq8`)—about 1 byte per dimension plus per-row overhead. This one is an
  estimate; confirm it with `pg_relation_size` on a loaded cluster.
- **Btrees**—PK plus one FK index.

Treat the totals as **±15%** until a real load is measured with `pg_total_relation_size`.

### Load-time model

Calibrated on Lab 1: the student notebook loads the 8 relational tables plus 14,235 vector rows
in **~4.5 min** through the AlloyDB Python Connector. The relational/vector split wasn't measured,
so `ANCHOR_RELATIONAL_S` brackets it at 60–120 s. That gives a vector rate in **CSV bytes per
second**, which is rescaled to the report rows' (slightly smaller) CSV size. ScaNN build time is
scaled linearly from 27–47 s at 13,439 rows.

**The cold-start budget**: Start Lab takes ~20 min, of which the cluster is ~9, which leaves
**~11 min** for everything else. The verdict column checks relational + vector + ScaNN against that.

In [ ]:
S = "9 · Size math"
CAL = {}

# ---- Measure 1: the existing profile text (gen checkpoint) -------------------------------
gen_cands = INV_CKPT[INV_CKPT.name.str.match(r"gen_players_v[\d.]+\.csv$")] if len(INV_CKPT) else INV_CKPT
GEN = None
if len(gen_cands):
    def _ver(n):
        return tuple(int(x) for x in re.findall(r"v([\d.]+)\.csv", n)[0].split("."))
    _gen_uri = max(gen_cands.uri, key=lambda u: _ver(u.rsplit("/", 1)[-1]))
    r = gcs("cat", _gen_uri, binary=True)
    if r.returncode == 0:
        GEN = pd.read_csv(io.BytesIO(r.stdout), low_memory=False)
        GEN["text"] = GEN["text"].fillna("").astype(str)
        GEN = GEN[~GEN.text.str.startswith("[[PROFILE_GENERATION_FAILED") & (GEN.text.str.strip() != "")]
        GEN["words"] = GEN.text.str.split().str.len()
        GEN["bytes"] = GEN.text.map(lambda s: len(s.encode("utf-8")))
        CAL["gen_source"] = _gen_uri.replace(GCS_PREFIX + "/", "")
        CAL["gen_rows"] = len(GEN)
        CAL["words_mean"] = float(GEN.words.mean())
        CAL["bytes_per_word"] = float(GEN.bytes.sum() / GEN.words.sum())
        for c in ("in_tokens", "out_tokens", "thinking_tokens"):
            if c in GEN.columns:
                CAL[c] = float(pd.to_numeric(GEN[c], errors="coerce").mean())
        if {"out_tokens", "thinking_tokens"} <= set(GEN.columns):
            _vis = pd.to_numeric(GEN.out_tokens, errors="coerce") - pd.to_numeric(GEN.thinking_tokens, errors="coerce")
            CAL["visible_tokens_per_word"] = float(_vis.sum() / GEN.words.sum())
if "bytes_per_word" not in CAL:
    print("⚠️  no readable gen_players_*.csv under checkpoints/—using defaults for text size and tokens")
BYTES_PER_WORD = CAL.get("bytes_per_word", DEFAULT_BYTES_PER_WORD)

# ---- Measure 2: CSV bytes per vector row (stream the first records) ----------------------
csv.field_size_limit(64 * 1024 * 1024)
_prof_uri = f"{GCS_PREFIX}/players_profiles.csv.gz"
SAMPLE_N = 300
try:
    fh, close = gcs_stream(_prof_uri)
    rows_ = []
    with gzip.GzipFile(fileobj=fh) as gz:
        for row in csv.reader(io.TextIOWrapper(gz, encoding="utf-8", newline="")):
            rows_.append(row)
            if len(rows_) >= SAMPLE_N:
                break
    close()
    if rows_:
        vec_lens, txt_words, row_bytes_ = [], [], []
        for row in rows_:
            vec = max(row, key=lambda f: f.count(","))
            txt = max((f for f in row if f is not vec), key=len)
            nvals = vec.count(",") + 1
            vec_lens.append(len(vec.encode()) / nvals)
            txt_words.append(len(txt.split()))
            row_bytes_.append(sum(len(f.encode()) for f in row) + len(row))
        CAL["csv_chars_per_value"] = float(np.mean(vec_lens))
        CAL["anchor_csv_row_bytes"] = float(np.mean(row_bytes_))
        CAL["anchor_words"] = float(np.mean(txt_words))
        _obj = OBJ_BYTES.get(_prof_uri)
        if _obj:
            CAL["gzip_ratio"] = _obj / (CAL["anchor_csv_row_bytes"] * EXPECTED_ROWS["players"])
            CAL["profiles_gz_bytes"] = _obj
except Exception as exc:  # noqa: BLE001
    print(f"⚠️  could not sample {_prof_uri}: {type(exc).__name__}: {exc}")
# Sanity bounds: a measurement outside these is more likely a sampling artefact than reality.
if not 0.15 <= CAL.get("gzip_ratio", 0.45) <= 0.80:
    print(f"⚠️  measured gzip ratio {CAL.get('gzip_ratio'):.3f} is implausible "
          "(players_profiles.csv.gz row count may not be 13,439)—using 0.45")
    CAL.pop("gzip_ratio", None)
if not 1.0 <= CAL.get("visible_tokens_per_word", 1.35) <= 2.0:
    print(f"⚠️  measured visible tokens/word {CAL.get('visible_tokens_per_word'):.2f} is outside "
          f"1.0–2.0—using the default {TOKENS_PER_WORD_DEFAULT}")
    CAL.pop("visible_tokens_per_word", None)
CSV_CHARS_PER_VALUE = CAL.get("csv_chars_per_value", 11.0)
GZIP_RATIO = CAL.get("gzip_ratio", 0.45)
ANCHOR_CSV_ROW = CAL.get("anchor_csv_row_bytes",
                         CSV_CHARS_PER_VALUE * EMBED_DIMS + 250 * BYTES_PER_WORD + 20)

print("CALIBRATION")
for k, v in CAL.items():
    print(f"   {k:<28} {v:,.3f}" if isinstance(v, float) else f"   {k:<28} {v}")
print(f"   using bytes/word {BYTES_PER_WORD:.2f}, CSV chars/value {CSV_CHARS_PER_VALUE:.2f}, "
      f"gzip ratio {GZIP_RATIO:.3f}")


# ---- The per-row model -------------------------------------------------------------------
def align8(n):
    return int(math.ceil(n / 8) * 8)


def row_model(words):
    vec_raw = EMBED_DIMS * BYTES_PER_FLOAT + 8
    chunks = math.ceil(vec_raw / TOAST_CHUNK)
    toast = vec_raw + chunks * TOAST_TUPLE_OVH
    toast_idx = chunks * TOAST_IDX_PER_CHUNK
    text = words * BYTES_PER_WORD
    heap = align8(TUPLE_HDR + REPORT_META_BYTES + text + 4 + TOAST_POINTER) + LINE_PTR
    scann = EMBED_DIMS * 1 + SCANN_BYTES_PER_ROW_OVH
    btree = REPORT_BTREES * BTREE_BYTES_PER_ROW
    csv_row = CSV_CHARS_PER_VALUE * EMBED_DIMS + text + 40
    return {"words": words, "vector_raw": vec_raw, "toast_chunks": chunks, "toast": toast,
            "toast_idx": toast_idx, "heap": heap, "scann": scann, "btree": btree,
            "disk_total": toast + toast_idx + heap + scann + btree,
            "csv": csv_row, "csv_gz": csv_row * GZIP_RATIO}


RM = {w: row_model(w) for w in REPORT_WORDS}
print("\nBYTES PER REPORT ROW")
_lbl = {"vector_raw": "vector (pgvector, raw)", "toast": "  stored in TOAST (+chunk overhead)",
        "toast_idx": "  TOAST index", "heap": "heap tuple (meta + text + ptr)",
        "scann": "ScaNN sq8 (estimate)", "btree": f"{REPORT_BTREES} btree entries",
        "disk_total": "ON-DISK TOTAL", "csv": "CSV for COPY (6 sig. digits)", "csv_gz": "  gzipped (staging)"}
print(f"   {'':<38}" + "".join(f"{w:>12} words" for w in REPORT_WORDS))
for k, lab in _lbl.items():
    print(f"   {lab:<38}" + "".join(f"{RM[w][k]:>18,.0f}" for w in REPORT_WORDS))
print(f"   (vector = {EMBED_DIMS} x {BYTES_PER_FLOAT} B + 8 B header, {RM[REPORT_WORDS[0]]['toast_chunks']} TOAST chunks)")


# ---- Load-time model ---------------------------------------------------------------------
def vec_rate_rows_per_s(csv_row_bytes):
    if LOAD_ROWS_PER_S_OVERRIDE:
        return LOAD_ROWS_PER_S_OVERRIDE, LOAD_ROWS_PER_S_OVERRIDE
    rates = []
    for rel in ANCHOR_RELATIONAL_S:          # rel=60 -> slower vector rate (pessimistic)
        bytes_per_s = ANCHOR_VECTOR_ROWS * ANCHOR_CSV_ROW / (ANCHOR_NOTEBOOK_LOAD_S - rel)
        rates.append(bytes_per_s / csv_row_bytes)
    return min(rates), max(rates)


BUDGET_S = (COLD_START_MIN - CLUSTER_MIN) * 60


def load_estimate(rows, words):
    rm = RM[words]
    r_lo, r_hi = vec_rate_rows_per_s(rm["csv"])
    vec = (rows / r_hi, rows / r_lo)
    scann = tuple(s * rows / ANCHOR_SCANN_ROWS for s in ANCHOR_SCANN_S)
    rel = ANCHOR_RELATIONAL_S
    total = (rel[0] + vec[0] + scann[0], rel[1] + vec[1] + scann[1])
    verdict = ("fits" if total[1] <= BUDGET_S else
               "maybe" if total[0] <= BUDGET_S else "over")
    return vec, scann, total, verdict


# ---- Candidate corpora and lab slices ----------------------------------------------------
last3 = [s for s in COMPLETE_SEASONS if s > LATEST_COMPLETE - 3]
_rcx = RC.set_index("min minutes")
CANDIDATES = [
    ("Lab 1 today (player profiles, reference)", 13_439),
    (f"1 season >=900 min ({LATEST_COMPLETE}/{str(LATEST_COMPLETE + 1)[-2:]}), k=1",
     int(((PS.season == LATEST_COMPLETE) & (PS.minutes >= 900)).sum())),
    (f"last 3 complete seasons >=900 min, k=1",
     int((PS.season.isin(last3) & (PS.minutes >= 900)).sum())),
    ("all seasons >=900 min, k=1", TH[900]),
    ("all seasons >=900 min, k=2 capped", int(_rcx.loc[900, "k=2 capped"])),
    ("all seasons >=450 min, k=2 capped", int(_rcx.loc[450, "k=2 capped"])),
    ("FULL: all player-seasons, k=1", N_PS),
    ("FULL: all player-seasons, k=2 capped", int(_rcx.loc[1, "k=2 capped"])),
    ("FULL: all player-seasons, k=3 capped", int(_rcx.loc[1, "k=3 capped"])),
    ("outline floor", OUTLINE_PROMISE[0]),
    ("outline ceiling", OUTLINE_PROMISE[1]),
    ("round: 5K", 5_000), ("round: 10K", 10_000), ("round: 25K", 25_000), ("round: 50K", 50_000),
]

w_lo, w_hi = REPORT_WORDS
size_rows = []
for label, n in CANDIDATES:
    vec, scann, total, verdict = load_estimate(n, w_hi)
    size_rows.append({
        "slice": label, "rows": f"{n:,}",
        "on-disk": fmt_bytes(n * RM[w_hi]["disk_total"]),
        "staged .csv.gz": fmt_bytes(n * RM[w_hi]["csv_gz"]),
        "vector load": f"{vec[0] / 60:,.1f}–{vec[1] / 60:,.1f} min",
        "ScaNN": f"{scann[0] / 60:,.1f}–{scann[1] / 60:,.1f} min",
        "total incl. relational": f"{total[0] / 60:,.1f}–{total[1] / 60:,.1f} min",
        f"11-min budget": verdict,
    })
SIZE = pd.DataFrame(size_rows)
r_lo, r_hi = vec_rate_rows_per_s(RM[w_hi]["csv"])
print(f"\nLOAD RATE (calibrated): {r_lo:,.0f}–{r_hi:,.0f} vector rows/s at {w_hi} words "
      f"(~{r_lo * RM[w_hi]['csv'] / 1e6:,.1f}–{r_hi * RM[w_hi]['csv'] / 1e6:,.1f} MB/s of CSV through the Connector)")
print(f"cold-start budget after the cluster: {BUDGET_S // 60} min. "
      f"'fits' = high estimate inside it, 'maybe' = only the low estimate is, 'over' = neither.\n")
print(f"SLICE TOTALS (sized at {w_hi} words; {w_lo} words is "
      f"{(1 - RM[w_lo]['disk_total'] / RM[w_hi]['disk_total']):.1%} smaller on disk)")
show(SIZE)
print("\nNotes: on-disk = heap + TOAST + TOAST index + ScaNN + 2 btrees, excluding the relational "
      "tables. Load times assume the Lab 1 path (COPY over the Python Connector from Colab); a "
      "faster path would change these and should be measured, not assumed.")

hb(S, "calibration: " + ", ".join(f"{k}={v:,.3f}" if isinstance(v, float) else f"{k}={v}" for k, v in CAL.items()))
for w in REPORT_WORDS:
    rm = RM[w]
    hb(S, f"per row @{w} words: vector {rm['vector_raw']:,} B; TOAST {rm['toast']:,.0f}; heap {rm['heap']:,.0f}; "
          f"ScaNN {rm['scann']:,.0f}; on-disk total {rm['disk_total']:,.0f} B; CSV {rm['csv']:,.0f} B; gz {rm['csv_gz']:,.0f} B")
hb(S, f"load rate {r_lo:,.0f}–{r_hi:,.0f} rows/s; budget {BUDGET_S // 60} min")
for _, r in SIZE.iterrows():
    hb(S, f"{r.slice}: {r.rows} rows | {r['on-disk']} | gz {r['staged .csv.gz']} | load {r['total incl. relational']} | {r['11-min budget']}")
send_back(S)

---
## 10 · Report-generation cost

Arithmetic only—nothing here calls a model. `gemini-3.8-flash` at the **promotional** list
price of $0.75 input / $3.75 output per 1M tokens, **halved for batch** = $0.375 / $1.875.

Per report:

- **Input tokens**—the fact block plus instructions. Measured from the Lab 1 player prompts
  (`in_tokens` in Part 2's checkpoint) when readable, otherwise `INPUT_TOKENS_PER_REPORT_DEFAULT`.
  Those prompts were *grounded* per-player fact blocks; a per-season block may be shorter.
- **Output tokens**—120–180 words × visible tokens per word (measured from the same checkpoint,
  otherwise 1.35).
- **Reasoning tokens**—Gemini 3.x has no "off", and reasoning bills at the output rate. Shown
  both without reasoning and with the mean reasoning tokens Part 2 measured at `LOW`.

Also shown: what the same run costs **after the promotion** (Part 2 notes the 3.7-flash promo
doubles on 2027-01-01), the embedding cost at Part 2's `gemini-embedding-001` list price, and a
"what if" for Search grounding, which this corpus shouldn't need because the facts come from the
database.

In [ ]:
S = "10 · Generation cost"
TPW = CAL.get("visible_tokens_per_word", TOKENS_PER_WORD_DEFAULT)
IN_TOK = CAL.get("in_tokens", INPUT_TOKENS_PER_REPORT_DEFAULT)
REASON = CAL.get("thinking_tokens", REASONING_TOKENS_DEFAULT)
p_in = PRICE_IN_PER_M * BATCH_DISCOUNT
p_out = PRICE_OUT_PER_M * BATCH_DISCOUNT
pp_in, pp_out = (x * BATCH_DISCOUNT for x in PRICE_POSTPROMO)

print(f"per report: input {IN_TOK:,.0f} tok   output {w_lo}–{w_hi} words x {TPW:.2f} = "
      f"{w_lo * TPW:,.0f}–{w_hi * TPW:,.0f} tok   reasoning {REASON:,.0f} tok (when included)")
print(f"batch prices: promo ${p_in:.3f} in / ${p_out:.3f} out per 1M   "
      f"post-promo ${pp_in:.3f} / ${pp_out:.3f}\n")


def cost(n, words, reasoning, pin, pout):
    out_tok = words * TPW + reasoning
    return n * (IN_TOK * pin + out_tok * pout) / 1e6


def per_report(words, reasoning, pin, pout):
    return (IN_TOK * pin + (words * TPW + reasoning) * pout) / 1e6


print("PER REPORT (USD)")
for label, rsn in [("no reasoning", 0), (f"+{REASON:,.0f} reasoning", REASON)]:
    print(f"   {label:<22} promo ${per_report(w_lo, rsn, p_in, p_out):.5f}–${per_report(w_hi, rsn, p_in, p_out):.5f}"
          f"   post-promo ${per_report(w_lo, rsn, pp_in, pp_out):.5f}–${per_report(w_hi, rsn, pp_in, pp_out):.5f}")

cost_rows = []
for label, n in CANDIDATES:
    cost_rows.append({
        "slice": label, "reports": f"{n:,}",
        "tokens in (M)": round(n * IN_TOK / 1e6, 1),
        "tokens out (M)": f"{n * w_lo * TPW / 1e6:,.1f}–{n * (w_hi * TPW + REASON) / 1e6:,.1f}",
        "promo, no reasoning": f"${cost(n, w_lo, 0, p_in, p_out):,.0f}–${cost(n, w_hi, 0, p_in, p_out):,.0f}",
        "promo, + reasoning": f"${cost(n, w_lo, REASON, p_in, p_out):,.0f}–${cost(n, w_hi, REASON, p_in, p_out):,.0f}",
        "post-promo, + reasoning": f"${cost(n, w_lo, REASON, pp_in, pp_out):,.0f}–${cost(n, w_hi, REASON, pp_in, pp_out):,.0f}",
        "embeddings": f"${n * w_hi * TPW * PRICE_EMBED_PER_M / 1e6:,.2f}",
    })
COST = pd.DataFrame(cost_rows)
show(COST)
print(f"\nembeddings: {w_hi} words x {TPW:.2f} tok at ${PRICE_EMBED_PER_M}/1M (Part 2 list price, no batch discount applied)")
print(f"grounding what-if: +${PRICE_GROUNDING_PER_1K:.0f} per 1,000 reports at Part 2's rate—e.g. "
      f"+${TH[900] * PRICE_GROUNDING_PER_1K / 1000:,.0f} for the >=900 min k=1 corpus. Not included above.")

hb(S, f"per report: in {IN_TOK:,.0f} tok; out {w_lo * TPW:,.0f}–{w_hi * TPW:,.0f} tok; reasoning {REASON:,.0f}; "
      f"tok/word {TPW:.2f}; promo batch ${p_in:.3f}/${p_out:.3f}")
hb(S, f"per report USD: promo no-reasoning ${per_report(w_lo, 0, p_in, p_out):.5f}–${per_report(w_hi, 0, p_in, p_out):.5f}; "
      f"promo +reasoning ${per_report(w_lo, REASON, p_in, p_out):.5f}–${per_report(w_hi, REASON, p_in, p_out):.5f}")
for _, r in COST.iterrows():
    hb(S, f"{r.slice}: {r.reports} | promo {r['promo, no reasoning']} (+reasoning {r['promo, + reasoning']}) | "
          f"post-promo {r['post-promo, + reasoning']} | emb {r.embeddings}")
send_back(S)

---
## 11 · History tables—appearances, game events, games

Rows, shape, and size for the three history tables as they're scoped today. **Parquet size** is
measured by writing each table to the local runtime (`/content/cymbalgoal-sizing/parquet-measure/`)
with snappy, Part 1's checkpoint codec, and with zstd for comparison. Nothing is written to the
bucket. The source object's size in the bucket is shown alongside.

In [ ]:
S = "11 · History tables"
hist_rows = []
for t, dcol in [("appearances", "appearance_date"), ("game_events", "event_date"), ("games", "game_date")]:
    df = T[t]
    p_snappy = MEASURE_DIR / f"{t}.snappy.parquet"
    p_zstd = MEASURE_DIR / f"{t}.zstd.parquet"
    df.to_parquet(p_snappy, index=False, compression="snappy")
    df.to_parquet(p_zstd, index=False, compression="zstd")
    if t == "games":
        seasons_ = df.season
    else:
        seasons_ = df[["game_id"]].merge(g[["game_id", "season"]], on="game_id", how="left").season
    dr = df[dcol].dropna() if dcol in df.columns else pd.Series(dtype="datetime64[ns]")
    hist_rows.append({
        "table": t, "rows": f"{len(df):,}", "cols": df.shape[1],
        "seasons": f"{int(seasons_.min())}–{int(seasons_.max())}" if seasons_.notna().any() else "—",
        "rows/season": f"{len(df) / max(seasons_.nunique(), 1):,.0f}",
        "dates": f"{dr.min().date()} → {dr.max().date()}" if len(dr) else "—",
        "bucket object": f"{SRC[t].rsplit('/', 1)[-1]} {fmt_bytes(SRC_BYTES[t])}",
        "parquet snappy": fmt_bytes(p_snappy.stat().st_size),
        "parquet zstd": fmt_bytes(p_zstd.stat().st_size),
        "snappy B/row": f"{p_snappy.stat().st_size / max(len(df), 1):,.1f}",
        "in memory": fmt_bytes(df.memory_usage(deep=True).sum()),
    })
HIST = pd.DataFrame(hist_rows)
show(HIST)
if "game_events" in T:
    print("\ngame_events by event_type:")
    for k, v in T["game_events"].event_type.value_counts().items():
        print(f"   {str(k):<16}{v:>10,}")

for _, r in HIST.iterrows():
    hb(S, f"{r.table}: {r.rows} rows x {r.cols} cols | seasons {r.seasons} | {r.dates} | "
          f"bucket {r['bucket object']} | parquet snappy {r['parquet snappy']} ({r['snappy B/row']} B/row), zstd {r['parquet zstd']}")
send_back(S)

---
## 12 · Sizing handback

Everything from the **📋 SEND BACK** blocks above in one place. Copy the whole output of this cell
and paste it into the checklist. It's also saved to `/content/cymbalgoal-sizing/sizing_handback.txt`
(local runtime only).

The **read-only audit** at the bottom lists every bucket call this run made. All of them should
be `ls` or `cat`, and no Gemini or Vertex module should have been imported.

In [ ]:
lines = []
lines.append("=" * 78)
lines.append("CYMBALGOAL ALLOYDB WORKSHOP (TRACK A)—DATA SPIKE HANDBACK  (00-explore-and-size.ipynb)")
lines.append(f"run {datetime.now(timezone.utc).isoformat(timespec='seconds')}   "
             f"snapshot {SNAPSHOT_DATE} {SNAPSHOT_SHA256[:12]}")
lines.append("=" * 78)
for sec, ls_ in HB.items():
    lines.append("")
    lines.append(f"## {sec}")
    lines.extend(ls_)

# ---- read-only audit ---------------------------------------------------------------------
verbs = Counter(c.split()[2] for c in GCS_LOG)
writes = [c for c in GCS_LOG if c.split()[2] not in _READ_VERBS]
ai_mods = sorted(m for m in sys.modules if m.startswith(("google.genai", "vertexai", "google.cloud.aiplatform")))
lines.append("")
lines.append("## Read-only audit")
lines.append(f"bucket calls: {len(GCS_LOG)} ({', '.join(f'{v} {k}' for k, v in verbs.items())}); "
             f"writes: {len(writes)}")
lines.append(f"Gemini/Vertex modules imported: {', '.join(ai_mods) if ai_mods else 'none'}")
lines.append("timings: " + ", ".join(f"{k} {v:.0f}s" for k, v in TIMERS.items()))
lines.append("=" * 78)

text = "\n".join(lines)
print(text)
(BASE / "sizing_handback.txt").write_text(text)
print(f"\n[saved to {BASE / 'sizing_handback.txt'}—local runtime only]")

assert not writes, f"a write reached gcs(): {writes}"
assert not ai_mods, f"Gemini/Vertex imported: {ai_mods}"